# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [ ]:
# Paso 0: instalación, credenciales, descarga del PDF y cliente NVIDIA NIM  --> RESUELTO
# Ejecuta esta celda primero. Funciona igual en Google Colab y en local.

# 1) Instalar dependencias (en Colab/limpio). Si ya las tienes, no pasa nada.
%pip install -q --upgrade pypdf gradio openai python-dotenv requests

import os
import requests
from pypdf import PdfReader
from openai import OpenAI

# 2) Leer la API key de forma SEGURA (nunca la escribas en el código):
#    - En Google Colab: usa el panel "Secrets" (icono de llave) y crea
#      un secreto llamado NVIDIA_API_KEY con tu key.
#    - En local: crea un archivo .env con  NVIDIA_API_KEY="nvapi-...."
NVIDIA_API_KEY = None
try:
    # Opcion A: Google Colab Secrets
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except Exception:
    pass

if not NVIDIA_API_KEY:
    # Opcion B: archivo .env (ejecucion local)
    from dotenv import load_dotenv
    load_dotenv()
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

assert NVIDIA_API_KEY, (
    "No encontre la API key. En Colab crea el secreto NVIDIA_API_KEY (icono de llave); "
    "en local crea un archivo .env con NVIDIA_API_KEY=\"nvapi-....\""
)
print("API key cargada correctamente (no se imprime por seguridad).")

# 3) Descargar el paper automaticamente si no existe
PDF_PATH = "attention_is_all_you_need.pdf"
if not os.path.exists(PDF_PATH):
    print("Descargando el paper...")
    url = "https://arxiv.org/pdf/1706.03762"
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    r.raise_for_status()
    with open(PDF_PATH, "wb") as fh:
        fh.write(r.content)
    print("PDF descargado:", PDF_PATH)
else:
    print("PDF ya presente:", PDF_PATH)

# 4) Cliente compatible con OpenAI apuntando a NVIDIA NIM
MODEL_NAME = "nvidia/llama-3.1-nemotron-70b-instruct"  # modelo estable del free tier
# Alternativa (si tu cuenta tiene acceso al de clase):
# MODEL_NAME = "nvidia/nemotron-3-super-120b-a12b"

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)
print("Cliente NVIDIA NIM configurado. Modelo:", MODEL_NAME)

### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [ ]:
# Paso 1: extracción de texto del PDF  --> RESUELTO
def extract_text_from_pdf(pdf_path):
    """Lee un PDF y devuelve todo su texto como una sola cadena."""
    reader = PdfReader(pdf_path)
    pages_text = []
    for page in reader.pages:
        text = page.extract_text() or ""   # evita None en páginas sin texto
        pages_text.append(text)
    return "\n".join(pages_text)

document_text = extract_text_from_pdf(PDF_PATH)
print("Caracteres extraídos:", len(document_text))
print("\nPrimeros 500 caracteres:\n")
print(document_text[:500])

## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [ ]:
# Paso 3: función de chat con el documento  --> RESUELTO
def build_system_prompt(document_text):
    """System prompt que obliga a responder SOLO con base en el documento."""
    return (
        "Eres un asistente experto en el paper proporcionado. "
        "Responde ÚNICAMENTE con información que aparezca en el documento. "
        "Si la respuesta no está en el documento, responde exactamente: "
        "'No encuentro esa información en el documento.'\n\n"
        "=== INICIO DEL DOCUMENTO ===\n"
        f"{document_text}\n"
        "=== FIN DEL DOCUMENTO ==="
    )


def history_to_messages(history):
    """Convierte el historial de Gradio (lista de pares [user, assistant])
    al formato de mensajes con roles de la API."""
    messages = []
    for user_msg, assistant_msg in history:
        if user_msg:
            messages.append({"role": "user", "content": user_msg})
        if assistant_msg:
            messages.append({"role": "assistant", "content": assistant_msg})
    return messages


def chat_con_documento(message, history, document_text):
    """Responde en streaming fundamentándose en el documento."""
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    messages += history_to_messages(history)
    messages.append({"role": "user", "content": message})

    stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=1.0,   # valores recomendados para Nemotron 3 Super
        top_p=0.95,
        stream=True,
    )

    partial = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            partial += delta
            yield partial

## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [ ]:
# Paso 4: interfaz Gradio  --> RESUELTO
n_pages = len(PdfReader(PDF_PATH).pages)
print(f"Documento: {n_pages} páginas, {len(document_text)} caracteres.")

demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="📄 Asistente del paper 'Attention Is All You Need'",
    description=(
        "Haz preguntas sobre el paper. El asistente responde únicamente "
        "con base en el contenido del documento."
    ),
    additional_inputs=[
        # Textbox oculto que inyecta el documento en cada llamada
        gr.Textbox(value=document_text, visible=False),
    ],
    examples=[
        ["¿Cuál es la arquitectura principal propuesta en el paper?"],
        ["¿Qué es el mecanismo de atención?"],
        ["¿Cuántas capas tiene el encoder del modelo base?"],
    ],
)

# share=True genera un enlace público (ideal en Colab).
demo.launch(share=True, show_error=True)

## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [ ]:
# Paso 6 (adicional): A) Medición del contexto  --> RESUELTO
n_chars = len(document_text)
# Estimación simple: ~4 caracteres por token en inglés (regla práctica)
est_tokens = n_chars / 4

print(f"Caracteres del documento : {n_chars:,}")
print(f"Tokens estimados (~char/4): {est_tokens:,.0f}")
print()
print("¿Por qué caracteres y tokens NO son equivalentes?")
print(
    "Un token no es un carácter ni siempre una palabra: el tokenizador parte el\n"
    "texto en sub-palabras según su vocabulario. Palabras frecuentes son 1 token,\n"
    "palabras raras se dividen en varios, y los espacios/puntuación cuentan aparte.\n"
    "Por eso la relación carácter→token varía con el idioma y el contenido; 4 chars\n"
    "por token es solo una aproximación para inglés."
)

## Paso 5: Prueba y reflexión
 
Una vez que la interfaz esté funcionando, prueba estas preguntas:
 
1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*
 
Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*
 
Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?
 

## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?
2. ¿Por qué este ejercicio todavía no implementa RAG?
3. ¿Cómo comprobarían que la respuesta proviene del documento y no del
   conocimiento previo del modelo?
4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el
   prompt?

## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)

---
### Respuestas (reflexión)

1. **Limitación de enviar siempre el documento completo:** se consume muchísimo contexto (tokens) en cada pregunta → mayor latencia, mayor costo y, con documentos largos, se puede superar la ventana de contexto del modelo. Además el modelo recibe mucho texto irrelevante para la pregunta puntual.
2. **¿Por qué esto todavía no es RAG?** Porque no hay **recuperación**: no se dividen el documento en fragmentos ni se seleccionan los más relevantes para la pregunta. Simplemente inyectamos todo el texto en el prompt. RAG recuperaría solo los pasajes pertinentes mediante búsqueda semántica (embeddings) antes de llamar al modelo.
3. **¿Cómo comprobar que la respuesta viene del documento y no del conocimiento previo?** Haciendo la **pregunta trampa** ("¿Qué es GPT-4?", que no está en el paper): si respeta la instrucción responderá "No encuentro esa información en el documento". También se puede pedir que **cite la página o el fragmento** que respalda cada respuesta.
4. **Riesgos de insertar documentos de usuarios directamente en el prompt:** *prompt injection* (el documento podría contener instrucciones que secuestren al modelo), fuga de datos sensibles hacia el proveedor, y respuestas no fundamentadas si el documento es muy largo y el modelo ignora partes.